In [16]:
## RAGAS is a framework for evaluating the RAG pipeline.

## Metrics:
# 1. Context Relevancy: Measures how relevant the retrieved context is to the question asked.
#    It filters out irrelevant information from the retrieved context.
#    Calculated as: number of relevant sentences in context / total sentences in context.

# 2. Context Precision: Measures the signal-to-noise ratio of the retrieved context.
#    It evaluates whether the relevant chunks are ranked higher than irrelevant ones.
#    Calculated as: mean of precision@k for each relevant chunk in the ranked retrieved context,
#    where precision@k = number of relevant chunks in top-k / k.

# 3. Context Recall: Measures how much of the ground truth is captured in the retrieved context.
#    It checks if all the necessary information to answer the question was retrieved.
#    Calculated as: Recall@K = number of ground truth sentences attributable to context / total sentences in ground tru th.

# 4. Faithfulness: Measures how factually consistent the generated answer is with the retrieved context.
#    It ensures the answer does not contain information not present in the context (no hallucinations).
#    Calculated as: number of answer statements that can be inferred from context / total statements in answer.

# 5. Answer Relevancy: Measures how relevant the generated answer is to the original question.
#    It penalizes answers that are incomplete or contain redundant information.
#    Calculated as: mean cosine similarity between the original question and n questions
#    generated from the answer, using embeddings to capture semantic similarity.

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_postgres import PGVector
from langchain_community.document_loaders import DirectoryLoader
import os
from dotenv import load_dotenv

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents.base import Document
from dotenv import load_dotenv
from langchain_chroma import Chroma
import os
from euriai.langchain import create_chat_model
import time
from euriai.langchain import EuriaiEmbeddings

app_dir = os.path.join(os.getcwd(), "app")
load_dotenv(os.path.join(app_dir, ".env"))

api_key = os.getenv("key")

chat_model = create_chat_model(api_key=api_key, model="gpt-4.1-nano", temperature=0.7)
model = chat_model

embeddings = EuriaiEmbeddings(
    api_key=api_key,
    model="text-embedding-3-small"
)

In [18]:
loader = DirectoryLoader("./data", glob="**/*.txt")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)
chunks = text_splitter.split_documents(docs)

libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.


In [22]:
model.invoke("hi")

HTTPError: 403 Client Error: Forbidden for url: https://api.euron.one/api/v1/euri/chat/completions

In [19]:
# RAGAS expects a file_name dict as key
for document in docs:
    document.metadata["file_name"] = document.metadata["source"]

In [20]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import default_query_distribution
from ragas.testset.graph import NodeType
from ragas.testset.transforms.extractors import EmbeddingExtractor, SummaryExtractor
from ragas.testset.transforms.extractors.llm_based import NERExtractor, ThemesExtractor
from ragas.testset.transforms.filters import CustomNodeFilter
from ragas.testset.transforms.relationship_builders import CosineSimilarityBuilder, OverlapScoreBuilder
from ragas.testset.transforms.engine import Parallel
from ragas.utils import num_tokens_from_string

generator_llm = LangchainLLMWrapper(model)
generator_embeddings = LangchainEmbeddingsWrapper(embeddings)

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)

query_distribution = default_query_distribution(generator_llm)

# Custom transforms that skip HeadlinesExtractor/HeadlineSplitter.
# The default pipeline uses headlines only for docs >500 tokens, but the
# Euriai LLM often fails to return parseable headlines, crashing HeadlineSplitter.
# This replicates the medium-doc pipeline which is robust across all LLMs.
def filter_doc(node):
    return node.type == NodeType.DOCUMENT and num_tokens_from_string(node.properties["page_content"]) > 100

def filter_docs(node):
    return node.type == NodeType.DOCUMENT

summary_extractor = SummaryExtractor(llm=generator_llm, filter_nodes=filter_doc)
summary_emb_extractor = EmbeddingExtractor(
    embedding_model=generator_embeddings,
    property_name="summary_embedding",
    embed_property_name="summary",
    filter_nodes=filter_doc,
)
cosine_sim_builder = CosineSimilarityBuilder(
    property_name="summary_embedding",
    new_property_name="summary_similarity",
    threshold=0.5,
    filter_nodes=filter_doc,
)
ner_extractor = NERExtractor(llm=generator_llm)
ner_overlap_sim = OverlapScoreBuilder(threshold=0.01)
theme_extractor = ThemesExtractor(llm=generator_llm, filter_nodes=filter_docs)
node_filter = CustomNodeFilter(llm=generator_llm)

custom_transforms = [
    summary_extractor,
    node_filter,
    Parallel(summary_emb_extractor, theme_extractor, ner_extractor),
    Parallel(cosine_sim_builder, ner_overlap_sim),
]

testset = generator.generate_with_langchain_docs(
    documents=docs,
    testset_size=8,
    query_distribution=query_distribution,
    transforms=custom_transforms,
    raise_exceptions=False,
)

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]c:\EGA\code\RAG\Udemy_RAG_1\Udemy-Advanced-LangChain\.venv\Lib\site-packages\euriai\langchain.py:323: RuntimeWarning: coroutine 'AsyncCallbackManagerForLLMRun.on_llm_error' was never awaited
  run_manager.on_llm_error(e)
Generating Samples: 100%|██████████| 9/9 [02:52<00:00, 19.20s/it]


Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]c:\EGA\code\RAG\Udemy_RAG_1\Udemy-Advanced-LangChain\.venv\Lib\site-packages\euriai\langchain.py:323: RuntimeWarning: coroutine 'AsyncCallbackManagerForLLMRun.on_llm_error' was never awaited
  run_manager.on_llm_error(e)
Generating Samples: 100%|██████████| 9/9 [02:52<00:00, 19.20s/it]


ValidationError: 2 validation errors for TestsetSample
eval_sample.SingleTurnSample
  Input should be a valid dictionary or instance of SingleTurnSample [type=model_type, input_value=nan, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type
eval_sample.MultiTurnSample
  Input should be a valid dictionary or instance of MultiTurnSample [type=model_type, input_value=nan, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type

In [21]:
testset.to_pandas()

NameError: name 'testset' is not defined

In [ ]:
from langchain_openai.embeddings import OpenAIEmbeddings

from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI

embedding = OpenAIEmbeddings()
model = ChatOpenAI(model="gpt-4o-mini")

vectorstore = Chroma.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever()

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = PromptTemplate(template=template, input_variables=["context", "question"])

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [ ]:
# questions = testset.to_pandas()["question"].to_list()
# ground_truth = testset.to_pandas()["ground_truth"].to_list()

import pandas as pd

df = pd.read_csv("./questions_answers/qa.csv", delimiter=";")
questions = df["question"].tolist()
ground_truth = df["ground_truth"].tolist()

In [ ]:
from datasets import Dataset

data = {"question": [], "answer": [], "contexts": [], "ground_truth": ground_truth}

for query in questions:
    data["question"].append(query)
    data["answer"].append(rag_chain.invoke(query))
    data["contexts"].append(
        [doc.page_content for doc in retriever.invoke(query)]
    )

dataset = Dataset.from_dict(data)

In [ ]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import evaluate

from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    LLMContextRecall,
    LLMContextPrecisionWithReference,
    ContextRelevance,
)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

metrics = [
    ContextRelevance(),
    LLMContextPrecisionWithReference(),
    LLMContextRecall(),
    Faithfulness(),
    AnswerRelevancy(),
]

In [ ]:
result = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=eval_llm,
    embeddings=eval_embeddings
)

print(result)

In [ ]:
result.to_pandas()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

df = result.to_pandas()

heatmap_data = df[
    [
        "context_relevance",
        "llm_context_precision_with_reference",
        "llm_context_recall",
        "faithfulness",
        "answer_relevancy",
    ]
]

cmap = LinearSegmentedColormap.from_list("green_red", ["red", "green"])

plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_data, annot=True, fmt=".2f", linewidths=0.5, cmap=cmap)

plt.yticks(ticks=range(len(df["question"])), labels=df["question"], rotation=0)

plt.show()